# Per-Cohort MICE Imputation — Numeric & Categorical

Fills missing values in `Final_metabric_data.tsv` using MICE, applied **independently within each cohort** so that no cohort's data informs another's imputation.



In [8]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer # need to enable the imputer to use IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder

RANDOM_STATE = 42
MICE_MAX_ITER = 10

NUMERIC_IMPUTE = [
    'Tumor Size',
]

CATEGORICAL_IMPUTE = [
    'Type of Breast Surgery',
    'Primary Tumor Laterality',
    'Pam50 + Claudin-low subtype',
    'Cellularity',
    'Radio Therapy',
    'Inferred Menopausal State', 
    'Integrative Cluster', 
    'Hormone Therapy', 
    'HER2 Status', 
    'HER2 status measured by SNP6', 
    'Chemotherapy',
]

ALL_IMPUTE_COLS = NUMERIC_IMPUTE + CATEGORICAL_IMPUTE

## 1. Load data

In [ ]:
# load the same processed METABRIC dataset used in the EDA notebook
df = pd.read_csv('../data/processed_data/Final_metabric_data.tsv', sep='\t')
print(f'Shape: {df.shape}')
print('\nMissing values per column (only columns with any missing):')
missing = df.isna().sum()
print(missing[missing > 0].to_string())
print(f'\nCohorts: {sorted(df["Cohort"].dropna().unique())}')

## 2. Fit OrdinalEncoder on the full dataset

We fit on all cohorts combined so the integer codes are consistent across cohorts even though imputation runs per cohort.

In [ ]:
# fit on all cohorts combined so category codes stay consistent even though MICE runs per cohort
ordinal = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=np.nan,
    encoded_missing_value=np.nan,
)
ordinal.fit(df[CATEGORICAL_IMPUTE])

print('Category mappings:')
for col, cats in zip(CATEGORICAL_IMPUTE, ordinal.categories_):
    print(f'  {col}: {list(cats)}')

## 3. Per-cohort MICE imputation

For each cohort we:
1. Build a matrix of `[numeric | ordinal-encoded categorical]` (NaN preserved).
2. Fit + transform MICE on that cohort alone.
3. Round ordinal columns to the nearest valid integer index.
4. Inverse-transform ordinal columns back to original category strings.
5. Write results back into the dataframe.

In [11]:
df_out = df.copy()

n_num = len(NUMERIC_IMPUTE)
category_sizes = [len(cats) for cats in ordinal.categories_]

cohorts = sorted(df['Cohort'].dropna().unique())
print(f'Imputing cohorts: {cohorts}\n')

for cohort in cohorts:
    mask = df['Cohort'] == cohort
    idx  = df.index[mask]
    sub  = df.loc[mask]

    # --- build imputation matrix ---
    num_block = sub[NUMERIC_IMPUTE].to_numpy(dtype=float)
    cat_block = ordinal.transform(sub[CATEGORICAL_IMPUTE])
    full_mat  = np.hstack([num_block, cat_block])

    pre_missing = np.isnan(full_mat).sum()

    # --- MICE ---
    imputer = IterativeImputer(random_state=RANDOM_STATE, max_iter=MICE_MAX_ITER)
    imputed = imputer.fit_transform(full_mat)

    post_missing = np.isnan(imputed).sum()
    print(f'Cohort {cohort:3.0f} | rows={len(sub):4d} | '
          f'missing before={pre_missing:3d} | missing after={post_missing}')

    # --- write back numeric ---
    for j, col in enumerate(NUMERIC_IMPUTE):
        df_out.loc[idx, col] = imputed[:, j]

    # --- round, clip, inverse-transform categorical ---
    cat_imputed = imputed[:, n_num:]
    cat_rounded = np.round(cat_imputed).astype(int)
    for i, size in enumerate(category_sizes):
        cat_rounded[:, i] = np.clip(cat_rounded[:, i], 0, size - 1)

    cat_labels = ordinal.inverse_transform(cat_rounded.astype(float))
    for j, col in enumerate(CATEGORICAL_IMPUTE):
        df_out.loc[idx, col] = cat_labels[:, j]

print('\nDone.')

Imputing cohorts: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]

Cohort   1 | rows= 522 | missing before= 87 | missing after=0
Cohort   2 | rows= 288 | missing before= 30 | missing after=0
Cohort   3 | rows= 763 | missing before= 23 | missing after=0
Cohort   4 | rows= 238 | missing before= 84 | missing after=0
Cohort   5 | rows= 170 | missing before= 11 | missing after=0

Done.


## 4. Verify — no missing values remain in imputed columns

In [ ]:
# confirm no missing values remain in the imputed columns
print('Missing values in imputed columns after per-cohort MICE:')
print(df_out[ALL_IMPUTE_COLS].isna().sum().to_string())


print('\nValue counts per imputed categorical column:')
for col in CATEGORICAL_IMPUTE:
    print(f'\n  {col}:')
    print(df_out[col].value_counts().to_string())

## 5. Quick sanity check — original vs imputed rows

In [ ]:
# spot-check a few rows that had missing data before vs. after imputation
was_missing_any = df[ALL_IMPUTE_COLS].isna().any(axis=1)
changed_rows = df_out[was_missing_any][['Cohort'] + ALL_IMPUTE_COLS]
original_rows = df[was_missing_any][['Cohort'] + ALL_IMPUTE_COLS]

print(f'Rows that had at least one missing value: {was_missing_any.sum()}')
print('\nOriginal (first 10 rows with any missing):')
print(original_rows.head(10).to_string())
print('\nImputed (same rows):')
print(changed_rows.head(10).to_string())

## 6. Save imputed dataset

In [ ]:
# write the imputed dataset for use in downstream modeling notebooks
out_path = '../data/processed_data/Final_metabric_data_mice_imputed.tsv'
df_out.to_csv(out_path, sep='\t', index=False)
print(f'Saved → {out_path}')
print(f'Shape: {df_out.shape}')